# Data Exploration

In [17]:
!pip install -q python-terrier ir_datasets pandas nltk

In [18]:
import pyterrier as pt
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

In [19]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

print(f"Loaded dataset: {dataset_name}")
irds_dataset = dataset.irds_ref()

Loaded dataset: irds:cord19/trec-covid


## Query Exploration
### Look at Verbosity Levels

In [20]:
topics_data = []
for query in irds_dataset.queries_iter():
    topics_data.append({
        # 'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })

df_topics = pd.DataFrame(topics_data)
print(f"Total number of queries: {len(df_topics)}")
df_topics.head(3)

Total number of queries: 50


,title,description,narrative
0,coronavirus origin,what is the origin of COVID-19,seeking range of information about the SARS-Co...
1,coronavirus response to weather changes,how does the coronavirus respond to changes in...,seeking range of information about the SARS-Co...
2,coronavirus immunity,will SARS-CoV2 infected people develop immunit...,seeking studies of immunity developed due to i...


### Average Length of the Query at each Verbosity Level Before and After Stopword Removal

In [21]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def count_words(text):
    if pd.isna(text): return 0
    return len(str(text).split())

def count_words_without_stopwords(text):
    if pd.isna(text): return 0
    words = str(text).lower().split()
    filtered_words = [w for w in words if w not in stop_words]
    return len(filtered_words)

In [22]:
verbosity_levels = ['title', 'description', 'narrative']

for level in verbosity_levels:
    df_topics[f'{level}_len'] = df_topics[level].apply(count_words)
    df_topics[f'{level}_len_no_stops'] = df_topics[level].apply(count_words_without_stopwords)


print("--- Average Query Length (INCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len = df_topics[f'{level}_len'].mean()
    print(f"{level.capitalize()}: {avg_len:.2f} words")

print("\n--- Average Query Length (EXCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len_no_stops = df_topics[f'{level}_len_no_stops'].mean()
    print(f"{level.capitalize()}: {avg_len_no_stops:.2f} words")

--- Average Query Length (INCLUDING Stopwords) ---
Title: 3.24 words
Description: 10.60 words
Narrative: 23.52 words

--- Average Query Length (EXCLUDING Stopwords) ---
Title: 2.74 words
Description: 5.56 words
Narrative: 14.74 words


## Relevance Judgments Exploration

In [23]:
qrels = dataset.get_qrels()

print(f"Total number of relevance judgments: {len(qrels)}")

print("\nRelevance Score Distribution:")
qrels['label'].value_counts()

Total number of relevance judgments: 69318

Relevance Score Distribution:


label
 0    42652
 2    15609
 1    11055
-1        2
Name: count, dtype: int64

## Document Exploration

In [24]:
doc_iter = irds_dataset.docs_iter()

sample_docs = []
for i, doc in enumerate(doc_iter):
    if i >= 5:
        break
    sample_docs.append({
        'docno': doc.doc_id,
        'title': doc.title,
        'abstract': doc.abstract[:300]
    })

df_docs = pd.DataFrame(sample_docs)
df_docs

,docno,title,abstract
0,ug7v899j,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...
3,2b73a28n,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide...
4,9785vg6d,Gene expression in epithelial cells in respons...,Respiratory syncytial virus (RSV) and pneumoni...


## Indexing

In [26]:
import os

if not os.path.exists("./index/data.properties"):
    indexer = pt.IterDictIndexer("./index")
    
    index_ref = indexer.index(
        ({
            "docno": d.doc_id,
            "text": (d.title or "") + " " + (d.abstract or "")
        }
        for d in irds_dataset.docs_iter())
    )
else:
    index_ref = "./index"

index = pt.IndexFactory.of(index_ref)

In [27]:
#loading the index
index = pt.IndexFactory.of("./cord19_index/data.properties")
#or
#index = pt.IndexFactory.of(index_ref)

20:27:37.716 [main] ERROR org.terrier.structures.Index -- Couldn't load an index structure called document
java.io.FileNotFoundException: /Users/nehiraltinkaya/Documents/GitHub/IR_Project/src/cord19_index/data.document.fsarrayfile (No such file or directory)
	at java.base/java.io.RandomAccessFile.open0(Native Method)
	at java.base/java.io.RandomAccessFile.open(RandomAccessFile.java:345)
	at java.base/java.io.RandomAccessFile.<init>(RandomAccessFile.java:259)
	at java.base/java.io.RandomAccessFile.<init>(RandomAccessFile.java:214)
	at java.base/java.io.RandomAccessFile.<init>(RandomAccessFile.java:127)
	at org.terrier.utility.io.LocalFileSystem$LocalRandomAccessFile.<init>(LocalFileSystem.java:63)
	at org.terrier.utility.io.LocalFileSystem.openFileRandom(LocalFileSystem.java:136)
	at org.terrier.utility.Files.openFileRandom(Files.java:400)
	at org.terrier.structures.collections.FSArrayFile.<init>(FSArrayFile.java:95)
	at org.terrier.structures.FSADocumentIndex.<init>(FSADocumentIndex.ja

JavaException: JVM exception occurred: java.lang.IllegalArgumentException: Could not load an index for ref ./cord19_index/data.properties, even though IndexLoader org.terrier.structures.IndexOnDisk$DiskIndexLoader could support that type of index. It may be your ref had a wrong location; Terrier logs may have more information.
java.lang.IllegalArgumentException: Could not load an index for ref ./cord19_index/data.properties, even though IndexLoader org.terrier.structures.IndexOnDisk$DiskIndexLoader could support that type of index. It may be your ref had a wrong location; Terrier logs may have more information.
	org.terrier.structures.IndexFactory.of(IndexFactory.java:135)
	org.terrier.structures.IndexFactory.of(IndexFactory.java:114)

## Preprocessing

In [29]:
#performing preprocessing on the queries
import re
import nltk
from nltk.stem import PorterStemmer
from spellchecker import SpellChecker
from nltk.corpus import stopwords
from nltk import pos_tag

nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')

stemmer = PorterStemmer()
spell = SpellChecker()
stop_words = set(stopwords.words('english'))

def preprocess_text(text, level):
    if pd.isna(text):
        return ""
    
    # lowercasing
    text = text.lower()
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # level 0: just lowercase + remove punctuation
    if level == 0:
        return text

    words = text.split()

    # level 1: POS filtering + stemming + stopword removal
    if level == 1:
        pos_tags = pos_tag(words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    # level 2: spell correction + POS filtering + stemming + stopword removal
    if level == 2:
        corrected_words = [spell.correction(token) or token for token in words]
        pos_tags = pos_tag(corrected_words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    return ' '.join(filtered_words)

print(verbosity_levels)

for level in verbosity_levels:
    print(f"\nPreprocessing with verbosity level: {level}")
    df_topics[f'preprocessed_title_{level}'] = df_topics['title'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_description_{level}'] = df_topics['description'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_narrative_{level}'] = df_topics['narrative'].apply(lambda x: preprocess_text(x, level))

print(df_topics.head(3))

['title', 'description', 'narrative']

Preprocessing with verbosity level: title


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


UnboundLocalError: cannot access local variable 'filtered_words' where it is not associated with a value